In [ ]:
#%pip install sktime==0.34.0   # brings the TSF loader
import pandas as pd
import numpy as np
import os
from sktime.datasets import load_tsf_to_dataframe
import matplotlib.pyplot as plt

from utils import *


# KDD

In [ ]:
raw_df, meta = load_tsf_to_dataframe(
    "kdd_cup_2018_dataset_without_missing_values.tsf",
    replace_missing_vals_with="NaN",
    return_type="default_tsf"
)
# Now raw_df *is* the 270‑row DataFrame:
print(raw_df.shape)      # (270, 6)
long_df = pd.concat(expand_row(r) for _, r in raw_df.iterrows())
print(long_df.head())

# ── 1.  Filter to PM 2.5 only ────────────────────────────────────────────────
pm25_df = long_df[long_df["pollutant"] == "PM2.5"].copy()

# ── 2.  Derive a 'season' column (N.‑hemisphere definition) ─────────────────
def month_to_season(month):
    if   month in (12, 1, 2):  return "Winter"
    elif month in (3, 4, 5):   return "Spring"
    elif month in (6, 7, 8):   return "Summer"
    else:                      return "Fall"

# apply to the timestamp level of the MultiIndex
months               = pm25_df.index.get_level_values("time").month
pm25_df["season"]    = [month_to_season(m) for m in months]

# ── 3.  Keep only the columns (city already present) ────────────────
pm25_df = pm25_df[["city","station","season","value"]]
pm25_df = pm25_df.reset_index()     
print(pm25_df.head())

weekly_df1 = process_weekly_df(pm25_df)
print(f"Shape of weekly DataFrame: {weekly_df1.shape}")

# BJ

In [ ]:
long_df = pd.DataFrame()
for filename in os.listdir("./bj"):
    df = pd.read_csv("bj/" + filename)
    df = df[['year', 'month', 'day', 'hour', 'PM2.5', 'station']].copy()
    long_df = pd.concat([long_df, df], ignore_index=True)
print(long_df.head())

# First, create a datetime column and season column
long_df['time'] = pd.to_datetime(long_df[['year', 'month', 'day', 'hour']])

# Create season column based on month
def month_to_season(month):
    if month in (12, 1, 2):  return "Winter"
    elif month in (3, 4, 5):   return "Spring"
    elif month in (6, 7, 8):   return "Summer"
    else:                      return "Fall"

long_df['season'] = long_df['month'].apply(month_to_season)
long_df['value'] = long_df['PM2.5']
long_df['city'] = 'Beijing'

pm25_df = long_df[['time', 'city', 'station', 'season', 'year', 'value']].copy()
print(pm25_df.head())

weekly_df2 = process_weekly_df(pm25_df)
print(f"Shape of weekly DataFrame: {weekly_df2.shape}")

In [ ]:
weekly_df = pd.concat([weekly_df1, weekly_df2], ignore_index=True)
print(f"Shape of weekly DataFrame: {weekly_df.shape}")
weekly_df.to_csv('air_quality.csv.zip', index=False, compression='zip')